**Question** Which model families solve failure, root-cause, propagation, and time-to-failure tasks on the same split?
**Method** Load the shared benchmark runner, reuse repository model classes, and report train-only classical baselines alongside the committed TGNN checkpoint.
**Takeaway** Model choice is an empirical comparison; graph capability is valuable only where the measured task requires it.

In [ ]:
from pathlib import Path
import sys, json, os
import pandas as pd
ROOT=Path.cwd(); sys.path.extend([str(ROOT),str(ROOT/'notebooks'),str(ROOT/'python-ml')])
RETRAIN=False
from notebooks.scripts.run_experiments import run
result=run(retrain=RETRAIN, reduced=os.environ.get('AEGIS_REDUCED')=='1')
pd.DataFrame(result['models']).T

### What this tells us
Every reported number in this table came from a model fit or checkpoint inference in the runner. Classical models use the same train/test episode split; `null` root-cause entries mean that the imported baseline has no root-cause head, not that a score was guessed.

In [ ]:
from models.tgnn.model import TGNNModel
from models.lstm.model import LSTMBaseline
from notebooks.common.data import load_dataset, split_arrays
data=split_arrays(load_dataset()); n=data['train']['x'].shape[2]; d=data['train']['x'].shape[-1]
architectures=[]
for name, model in [('GATv2+GRU TGNN', TGNNModel(in_dim=d,hidden_dim=64,num_nodes=n)), ('LSTM no graph', LSTMBaseline(num_nodes=n,feature_dim=d,hidden_dim=64,num_layers=1))]: architectures.append({'model':name,'parameters':sum(p.numel() for p in model.parameters()),'input_shape':str(tuple(data['train']['x'].shape[1:]))})
# The repository exposes TGNN's GATv2 layer as its spatial abstraction; no separate GCN/GATv2-last-step training implementation exists.
architectures.append({'model':'GCN+GRU ablation','parameters':None,'input_shape':'not implemented in python-ml; no fabricated score'})
architectures.append({'model':'GATv2 last-step ablation','parameters':None,'input_shape':'not implemented in python-ml; no fabricated score'})
display(pd.DataFrame(architectures))
print('RETRAIN:', RETRAIN, 'training metadata:', result['training'])

### What this tells us
Parameter counts are obtained from instantiated repository classes. The requested GCN+GRU and GATv2-last-step ablations are marked unavailable because this checkout does not expose those implementations; inventing scores would violate the experiment contract.

In [ ]:
grid=pd.DataFrame([{'model':'Logistic Regression','grid':'C in {0.1, 1, 10}; solver=lbfgs'}, {'model':'SVM','grid':'C in {0.1, 1, 10}; kernel=rbf'}, {'model':'Random Forest','grid':'n_estimators in {50, 100}; max_depth in {None, 8}'}, {'model':'XGBoost','grid':'n_estimators in {50, 100}; max_depth in {3, 6}'}, {'model':'TGNN','grid':'hidden_dim in {32, 64}; lr in {0.001, 0.003}; early stopping on validation'}])
display(grid)

### What this tells us
This is the pre-registered search space, not a claim that every candidate was selected. The default runner keeps execution short; setting `RETRAIN=True` is the explicit path for a fresh neural fit, and test data remains untouched during selection.